# MedMNIST EDA — Slice 2
Lightweight exploratory analysis of MedMNIST datasets.  
No torch. No tensors. Numpy + matplotlib only.

In [ ]:
# ─── Section 1: Imports and Config ────────────────────────────────────────────
import os
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend; change to TkAgg for local use
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style='whitegrid', font_scale=0.9)
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

# ── Project root on path ──────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from qcore.data.registry import get_dataset

# ── SET THIS ──────────────────────────────────────────────────────────────────
DATASET_NAME = "pneumoniamnist"   # switch to "pathmnist" to change dataset
SPLIT        = "train"
SEED         = 42

np.random.seed(SEED)
print(f'Dataset : {DATASET_NAME}')
print(f'Split   : {SPLIT}')
print(f'Seed    : {SEED}')

In [ ]:
# ─── Section 2: Load Dataset ──────────────────────────────────────────────────
dataset = get_dataset(DATASET_NAME, SPLIT)

print(f'Dataset name : {dataset.dataset_name}')
print(f'Split        : {dataset.split}')
print(f'Task         : {dataset.task}')
print(f'N classes    : {dataset.n_classes}')
print(f'Label map    : {dataset.label_map}')
print(f'Length       : {len(dataset)}')

In [ ]:
# ─── Section 3: Class Distribution ───────────────────────────────────────────
print('=== Class Distribution ===')

# Collect all labels
all_labels = np.array([dataset[i][1] for i in range(len(dataset))])

unique_classes, counts = np.unique(all_labels, return_counts=True)
total  = counts.sum()
pcts   = counts / total * 100
imbalance_ratio = counts.max() / counts.min()

print(f'  Total samples     : {total}')
print(f'  Imbalance ratio   : {imbalance_ratio:.2f}x  (max/min class)')
print()
print(f'  {"Class":<6}  {"Name":<40}  {"Count":>7}  {"Pct":>7}')
print(f'  {"-"*6}  {"-"*40}  {"-"*7}  {"-"*7}')
for cls, cnt, pct in zip(unique_classes, counts, pcts):
    name = dataset.label_map.get(int(cls), str(cls))
    print(f'  {cls:<6}  {name:<40}  {cnt:>7}  {pct:>6.1f}%')

if imbalance_ratio > 5:
    print(f'\n  [WARN] Imbalance ratio {imbalance_ratio:.1f}x > 5 — consider weighted sampling.')
else:
    print(f'\n  [OK] Imbalance ratio {imbalance_ratio:.2f}x within acceptable range (<= 5).')

# ── Bar chart ─────────────────────────────────────────────────────────────────
class_names = [dataset.label_map.get(int(c), str(c)) for c in unique_classes]
fig, ax = plt.subplots(figsize=(max(6, len(unique_classes) * 1.2), 4))
bars = ax.bar(class_names, counts, color='steelblue', edgecolor='k', linewidth=0.6)
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + total * 0.003,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_title(f'{DATASET_NAME} — {SPLIT} class distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Sample count')
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', f'{DATASET_NAME}_class_dist.png'), dpi=120)
plt.show()
print(f'  Plot saved → reports/{DATASET_NAME}_class_dist.png')

In [ ]:
# ─── Section 4: Random Sample Visualization ───────────────────────────────────
print('=== Random Sample Grid (3x3) ===')

rng = np.random.default_rng(seed=SEED)
indices = rng.choice(len(dataset), size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, idx in zip(axes.flat, indices):
    image, label = dataset[idx]
    class_name   = dataset.label_map.get(label, str(label))

    # Grayscale: (H, W) or (H, W, 1) → squeeze to 2D
    if image.ndim == 2 or (image.ndim == 3 and image.shape[2] == 1):
        ax.imshow(image.squeeze(), cmap='gray', vmin=0, vmax=1)
    else:
        # RGB: (H, W, 3)
        ax.imshow(np.clip(image, 0, 1))

    ax.set_title(class_name, fontsize=8)
    ax.axis('off')

plt.suptitle(f'{DATASET_NAME} — random samples ({SPLIT})', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', f'{DATASET_NAME}_samples.png'), dpi=120)
plt.show()
print(f'  Plot saved → reports/{DATASET_NAME}_samples.png')

In [ ]:
# ─── Section 5: Image Statistics ─────────────────────────────────────────────
print('=== Image Statistics ===')

N_SAMPLE = min(200, len(dataset))
sample_indices = np.random.choice(len(dataset), size=N_SAMPLE, replace=False)
sample_images  = np.stack([dataset[i][0] for i in sample_indices])

print(f'  Sample size : {N_SAMPLE}')
print(f'  Image shape : {sample_images[0].shape}')
print(f'  dtype       : {sample_images.dtype}')
print(f'  min         : {sample_images.min():.6f}')
print(f'  max         : {sample_images.max():.6f}')
print(f'  mean        : {sample_images.mean():.6f}')
print(f'  std         : {sample_images.std():.6f}')

# ── Hard assertions ───────────────────────────────────────────────────────────
if sample_images.dtype != np.float32:
    raise AssertionError(f'[FAIL] Expected float32, got {sample_images.dtype}')
print('  [OK] dtype == float32')

if sample_images.min() < 0.0 or sample_images.max() > 1.0:
    raise AssertionError(
        f'[FAIL] Values outside [0,1]: min={sample_images.min():.6f}, max={sample_images.max():.6f}'
    )
print('  [OK] all values in [0, 1]')

In [ ]:
# ─── Section 6: Channel Analysis ─────────────────────────────────────────────
print('=== Channel Analysis ===')

img0 = sample_images[0]

if img0.ndim == 2 or (img0.ndim == 3 and img0.shape[2] == 1):
    n_channels = 1
    print('  Grayscale — single channel confirmed.')
elif img0.ndim == 3 and img0.shape[2] == 3:
    n_channels = 3
    print('  RGB — 3 channels.')
    channel_names = ['Red', 'Green', 'Blue']
    for ch, name in enumerate(channel_names):
        ch_data = sample_images[:, :, :, ch] if sample_images.ndim == 4 else sample_images[..., ch]
        print(f'    {name}: mean={ch_data.mean():.4f}  std={ch_data.std():.4f}')
else:
    n_channels = img0.shape[-1] if img0.ndim == 3 else 1
    print(f'  Channels: {n_channels}  (shape: {img0.shape})')

print(f'  Image dimensions : {img0.shape}')

In [ ]:
# ─── Section 7: GO / NO-GO Summary ───────────────────────────────────────────
print('=== GO / NO-GO Summary ===')
print()

# Evaluate flags
imbalance_warning   = imbalance_ratio > 5
shape_consistent    = len(set(dataset[i][0].shape for i in range(min(20, len(dataset))))) == 1
label_type_valid    = isinstance(dataset[0][1], int)
value_range_valid   = (sample_images.min() >= 0.0) and (sample_images.max() <= 1.0)
ready_for_training  = (not imbalance_warning) and shape_consistent and label_type_valid and value_range_valid

def _yn(flag):   return 'YES' if flag else 'NO'
def _warn(flag): return ' ⚠️' if flag else ''

print(f'  Class imbalance warning (ratio > 5) : {_yn(imbalance_warning)}{_warn(imbalance_warning)}')
print(f'  Image shape consistent              : {_yn(shape_consistent)}')
print(f'  Label type valid (int)              : {_yn(label_type_valid)}')
print(f'  Value range valid [0, 1]            : {_yn(value_range_valid)}')
print(f'  Ready for training baseline         : {_yn(ready_for_training)}')
print()
# Static architectural confirmation — no torch code required
print('  Dataset ready for future Torch DataLoader integration : YES')
print('  (Adapter returns plain float32 numpy arrays and int labels')
print('   directly compatible with torch.utils.data.Dataset.__getitem__.)')
print()
if ready_for_training:
    print('>>> GO — dataset is valid. Ready to proceed to classification baseline.')
else:
    print('>>> NO-GO — review warnings above before proceeding.')
print()
print('=== NOTEBOOK RUN COMPLETE ===')